In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import csv
from functools import partial

## Load full GHC results

Load all GHC csv files and insert derived columns.

In [ ]:
def selectMedians(df):
    meds = []
    for n in df['cores'].unique():
        runs = df[df['cores']==n]
        med = runs[runs[' total']==runs[' total'].median()]
        # NB this median filtering might not work for an even number of iterations...
        # I think pandas will report the mean between the two medians.
        meds.append(med)
    return pd.concat(meds)

def parseBench(b):
    pars = selectMedians(pd.read_csv(f'i5/ghc/par/{b}.log'))
    baseline = selectMedians(pd.read_csv(f'i5/ghc/seq/{b}.log'))[' total'].iloc[0]
    pars['speedup'] = baseline/pars[' total']
    pars['%ideal'] = pars['speedup']/pars['cores']
    pars['bench'] = b
    return pars

def exportBenchCsvs(df, basename):
    benches = np.unique(df['bench'])
    for b in benches:
        df[df['bench']==b].to_csv(f'{basename}_{b}.csv', index=False, na_rep='nan')

In [ ]:
benches=!for f in i5/ghc/par/*.log; do echo $(basename -s '.log' $f); done
df = pd.concat([parseBench(b) for b in benches])

In [ ]:
display(px.line(df, x='cores', y='speedup', color='bench'))
display(px.line(df, x='cores', y=' productivity', color='bench'))

exportBenchCsvs(df,'i5/i5_summary')